# Preprocessing 1: Data cleaning and labels
**Input:** the MVSA-Single dataset (see `data/README.md`). **Output:** one clean table of posts for the next notebook.

| Step | What it does |
|---|---|
| 0 | Setup and configuration |
| 1 | Load posts and labels |
| 2 | **Missing values**: audit and handle missing, corrupt and duplicate data |
| 3 | Derive one label per post (standard MVSA rule) and tag agreement groups |
| 4 | Clean the text (URLs, mentions, hashtags, emojis) |

## Step 0: Setup
Settings and shared functions are in `src/mvsa/preprocessing.py`. The three you might change are `DATA_ROOT`, `OUT_DIR` and
`RUN_CLIP`. `emoji`, `wordsegment` and `ftfy` are small helper libraries. If they can't be installed, the code switches to
simpler built-in fallbacks and still runs.

In [1]:
# Make the shared code in src/ importable, wherever the repository folder is
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src", "mvsa")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from mvsa.preprocessing import *

Internet: on
GPU: Tesla T4
emoji: True | wordsegment: True | ftfy: True

⏱ Step 0 (setup) took 11.1s


## Step 1: Load posts and labels
`labelResultAll.txt` has one line per post: `ID<TAB>text_label,image_label`. The first line is a header.
The text and image for post `ID` are stored as `data/ID.txt` and `data/ID.jpg`.

In [2]:
def find_label_file(root):
    """Search the attached dataset for the label file, since the mount path can vary."""
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.lower() == "labelresultall.txt":
                return os.path.join(dirpath, f)
    raise FileNotFoundError(f"labelResultAll.txt not found under {root}. Is the MVSA-Single dataset attached?")

LABEL_FILE = find_label_file(DATA_ROOT)
DATA_DIR = os.path.join(os.path.dirname(LABEL_FILE), "data")

def parse_label_file(path):
    """Read the label file into a table with one row per post: id, text_label, image_label."""
    rows = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line or not line[0].isdigit():      # skips the header line
                continue
            tok = re.split(r"[,\t ]+", line)
            rows.append({"id": tok[0],
                         "text_label":  tok[1].lower() if len(tok) > 1 else None,
                         "image_label": tok[2].lower() if len(tok) > 2 else None})
    return pd.DataFrame(rows)

df = parse_label_file(LABEL_FILE)
REPORT["posts_in_label_file"] = len(df)
print(f"Label file: {LABEL_FILE}\nPosts listed: {len(df)}")
print(df.head())

Label file: /kaggle/input/datasets/vincemarcs/mvsasingle/MVSA_Single/labelResultAll.txt
Posts listed: 4869
  id text_label image_label
0  1    neutral    positive
1  2    neutral    positive
2  3    neutral    positive
3  4   positive    positive
4  5   positive    positive


## Step 2: Missing values (and other broken data)
In a tabular dataset, "missing values" means empty cells. In a multimodal dataset, a value can be missing
or unusable in several ways, so each one is checked separately:

| Problem | How it's detected | What we do | Why |
|---|---|---|---|
| Label missing or not one of pos/neu/neg | value check | **drop** | A sample with no valid target can't be used for supervised learning |
| Text file missing | file doesn't exist | **drop** | There is no text to embed |
| Text file empty | empty after `strip()` | **keep and flag** | An empty caption is real social-media behaviour. The image can still carry the sentiment |
| Image missing | file doesn't exist | **drop** | CLIP needs an image. Missing modalities are simulated later, in a controlled way |
| Image corrupt or truncated | `PIL.verify()` + `load()` fail | **drop** | It would crash or corrupt feature extraction |
| Exact duplicate (same text *and* same image bytes) | text + MD5 of the image | **keep the first copy** | Copies would be counted twice and could end up in both train and test |
| Garbled characters (e.g. `â€™`) | decoding fallback + `ftfy` | **repair** | Mis-decoded characters break tokenisation |

We **don't impute** (fill in) missing text or images. Imputing raw text or pixels isn't meaningful, and
the model notebook's robustness test masks one modality with its *training-set mean embedding*, which
is the right place to deal with a missing modality. Imputation therefore happens at the feature level, at
evaluation time, rather than here.

All 4,869 posts in this copy have both a text and an image, so most counts below should be 0.
Checking is still part of a reproducible pipeline: it proves the data is clean instead of assuming it.

In [3]:
def read_text(post_id):
    """Read one caption, returning (text, status), where status is ok, empty or missing_file."""
    path = os.path.join(DATA_DIR, f"{post_id}.txt")
    if not os.path.exists(path):
        return None, "missing_file"
    raw = open(path, "rb").read()
    for enc in ("utf-8", "cp1252", "latin-1"):        # latin-1 always succeeds, so this always returns
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    if ftfy is not None:
        text = ftfy.fix_text(text)                     # repairs mojibake such as "donâ€™t" -> "don't"
    text = text.strip()
    return text, ("ok" if text else "empty")

def inspect_image(post_id):
    """Check that one image exists and decodes, and return its size and MD5 for duplicate detection."""
    path = next((os.path.join(DATA_DIR, f"{post_id}{e}") for e in (".jpg", ".jpeg", ".png")
                 if os.path.exists(os.path.join(DATA_DIR, f"{post_id}{e}"))), None)
    if path is None:
        return {"image_path": None, "image_status": "missing_file"}
    with open(path, "rb") as f:
        data = f.read()                                # read the file once; everything below uses these bytes
    try:
        with Image.open(BytesIO(data)) as im:
            im.verify()                                # checks the file structure
        with Image.open(BytesIO(data)) as im:
            im.load()                                  # decodes the pixels, which catches truncated files
            w, h = im.size
    except Exception:
        return {"image_path": path, "image_status": "corrupt"}
    return {"image_path": path, "image_status": "ok", "width": w, "height": h,
            "image_md5": hashlib.md5(data).hexdigest()}

with ThreadPoolExecutor(WORKERS) as ex:               # files are read in parallel: several times faster
    texts = list(ex.map(read_text, df["id"]))
    img_info = pd.DataFrame(list(ex.map(inspect_image, df["id"])), index=df.index)
df["text_raw"]    = [t for t, _ in texts]
df["text_status"] = [s for _, s in texts]
df = pd.concat([df, img_info], axis=1)
df["label_valid"] = df["text_label"].isin(LABELS) & df["image_label"].isin(LABELS)

print("Missing cells per column (before handling):")
print(df.isna().sum().to_string())
tick("Step 2a (reading texts + checking images)")

Missing cells per column (before handling):
id              0
text_label      0
image_label     0
text_raw        0
text_status     0
image_path      0
image_status    0
width           0
height          0
image_md5       0
label_valid     0

⏱ Step 2a (reading texts + checking images) took 81.1s


In [4]:
n0 = len(df)
audit = []
def drop(mask, issue, why):
    """Remove the flagged posts and record how many went and why, for the audit table."""
    global df
    audit.append({"issue": issue, "posts": int(mask.sum()), "action": "dropped", "reason": why})
    df = df[~mask].copy()

drop(~df["label_valid"],                        "invalid or missing label", "no valid target")
drop(df["text_status"].eq("missing_file"),      "text file missing",        "nothing to embed")
drop(df["image_status"].eq("missing_file"),     "image file missing",       "nothing to embed")
drop(df["image_status"].eq("corrupt"),          "image corrupt/truncated",  "cannot be decoded")
dup = df.duplicated(subset=["text_raw", "image_md5"], keep="first")
drop(dup,                                        "exact duplicate post",     "would be counted twice / leak")

df["text_empty_raw"] = df["text_status"].eq("empty").astype(int)
audit.append({"issue": "text file empty", "posts": int(df["text_empty_raw"].sum()),
              "action": "kept + flagged", "reason": "image can still carry the sentiment"})
audit = pd.DataFrame(audit)
print(audit.to_string(index=False))
print(f"\n{n0} -> {len(df)} posts after handling missing and broken data")

REPORT["missing_value_audit"] = audit.to_dict(orient="records")
REPORT["posts_after_missing_handling"] = len(df)
tick("Step 2b (handling)")

                   issue  posts         action                              reason
invalid or missing label      0        dropped                     no valid target
       text file missing      0        dropped                    nothing to embed
      image file missing      0        dropped                    nothing to embed
 image corrupt/truncated      0        dropped                   cannot be decoded
    exact duplicate post     30        dropped       would be counted twice / leak
         text file empty      0 kept + flagged image can still carry the sentiment

4869 -> 4839 posts after handling missing and broken data

⏱ Step 2b (handling) took 0.0s


## Step 3: One label per post, plus agreement tags
This is the standard MVSA rule:
* text and image labels agree → use that label (**agreement**)
* one label is neutral → use the non-neutral one (**soft mismatch**)
* one is positive and the other negative → no single label (**hard mismatch**)

The standard rule *discards* hard mismatches. Here they're **set aside** in a separate file instead, so they
can be evaluated separately later. They are never used for training.
The `mismatch_type` tag is kept on every post so results can be reported per group.

On the full data: 4,484 labelled posts (2,670 / 468 / 1,346 pos / neu / neg) and 355 hard mismatches.

In [5]:
def mismatch_type(t, v):
    """Tag a post by how its text and image labels relate."""
    if t == v:
        return "agreement"
    return "hard_mismatch" if {t, v} == {"positive", "negative"} else "soft_mismatch"

def standard_label(t, v):
    """Apply the standard MVSA rule to get one label; None means a hard mismatch."""
    if t == v:      return t
    if t == "neutral": return v
    if v == "neutral": return t
    return None                                        # hard mismatch

df["mismatch_type"] = [mismatch_type(t, v) for t, v in zip(df.text_label, df.image_label)]
df["label"]         = [standard_label(t, v) for t, v in zip(df.text_label, df.image_label)]

print(df["mismatch_type"].value_counts().to_string())
print("\nLabel distribution (hard mismatches excluded):")
print(df["label"].value_counts().to_string())
REPORT["mismatch_counts"] = df["mismatch_type"].value_counts().to_dict()
REPORT["label_counts"]    = df["label"].value_counts().to_dict()
tick("Step 3")

mismatch_type
agreement        2576
soft_mismatch    1908
hard_mismatch     355

Label distribution (hard mismatches excluded):
label
positive    2670
negative    1346
neutral      468

⏱ Step 3 took 0.0s


## Step 4: Text cleaning
Cleaning is applied in this order (the order matters):
1. **HTML entities**: `&amp;` → `&`
2. **URLs removed**: `http://t.co/...` links carry no sentiment, and the link *count* is kept as a feature
3. **Mentions** → `@user`: the specific username is noise, but the fact that someone is addressed is kept
4. **Retweet prefix** `RT @user:` removed
5. **Hashtags segmented**: `#unicornNation` → `unicorn Nation`, `#happyfriday` → `happy friday`. A lot of
   the sentiment in tweets is in hashtags, and CLIP/TF-IDF can't read run-together words
6. **Emojis → words**: 😂 → `face with tears of joy`. CLIP's tokenizer handles emojis poorly, and the words carry the meaning
7. **Elongations shortened**: `soooo` → `soo`. This keeps the emphasis but merges the many spellings into one
8. Whitespace normalised

Case and punctuation are **kept on purpose**. `!!!` and ALL CAPS signal sentiment, CLIP was trained on natural
text, and TF-IDF lowercases by itself. **Stop-words aren't removed** either, because standard lists include
`not`, `no` and `never`, which reverse sentiment.

If a caption is empty after cleaning (e.g. it was only a link), it becomes the placeholder `"no text"` and the
`text_empty` flag is set, so the model can tell that case apart.

In [6]:
df["text_clean"] = df["text_raw"].apply(clean_text)
df["text_empty"] = df["text_clean"].eq("").astype(int)
df.loc[df["text_empty"] == 1, "text_clean"] = EMPTY_PLACEHOLDER
REPORT["text_empty_after_cleaning"] = int(df["text_empty"].sum())

pd.set_option("display.max_colwidth", 90)
changed = df[df["text_raw"] != df["text_clean"]]
print(f"{len(changed)} of {len(df)} texts changed by cleaning; "
      f"{df['text_empty'].sum()} were empty afterwards (set to '{EMPTY_PLACEHOLDER}').\n")
print(changed[["text_raw", "text_clean"]].head(8).to_string(index=False))
tick("Step 4 (text cleaning)")

4199 of 4839 texts changed by cleaning; 45 were empty afterwards (set to 'no text').

                                                                                                                         text_raw                                                                                                     text_clean
                                                                                     How I feel today #legday #jelly #aching #gym                                                                      How I feel today leg day jelly aching gym
            grattis min griskulting!!!???? va bara tvungen oki s? sch ? @ingenkommeratttrodig #pig #happybday #wow #lovely #cut¡­              grattis min griskulting!!!???? va bara tvungen oki s? sch ? @user pig happy bday wow lovely cut¡­
                                                     RT @polynminion: The moment I found my favourite tV character. #PROFOUNDLOVE                                                    The moment

## Save for the next notebook

In [ ]:
save_state("posts_clean", df)
save_state("report", REPORT)